<a href="https://colab.research.google.com/github/omsawarkar/omsawarkar/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/omsawarkar/omsawarkar/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

My lane is content review-priority scoring. I will use the March 2026 partition of fact_content_daily_performance.

The expected source grain is one row per report date, pseudonymized client, and pseudonymized content item. I will verify this with a duplicate check. My planned analysis table will contain one row per client–content pair, aggregating the eligible March observations.

The intended decision point is after March data is available. The output will support an editor’s review queue, rather than prove that updating a page will improve traffic. June 2026 will remain unused for development. I will exclude unavailable measurements using their availability flags instead of treating their zero-filled values as genuine observations.

In [16]:
%pip -q install huggingface_hub duckdb

import os
from google.colab import userdata
from huggingface_hub import HfApi

# Read the private token without displaying it.
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

api = HfApi(token=os.environ["HF_TOKEN"])
repo = "FlyRank/internship-warehouse"

# Check access by listing file names, not downloading the dataset.
warehouse_files = api.list_repo_files(
    repo_id=repo,
    repo_type="dataset"
)

print("Connection successful.")
print("Number of repository files:", len(warehouse_files))
print("\nMarch 2026 performance files:")

march_files = [
    path for path in warehouse_files
    if "fact_content_daily_performance" in path
    and "sample" not in path.lower()
    and "month=2026-03" in path
    and path.endswith(".parquet")
]

for path in march_files[:5]:
    print(path)

if not march_files:
    print("No matching paths found. We need to check the folder names.")

Connection successful.
Number of repository files: 24

March 2026 performance files:
fact_content_daily_performance/month=2026-03/data_0.parquet


In [17]:
# Check whether the expected daily grain is unique.
grain_check = con.sql("""
    SELECT COUNT(*) AS duplicate_key_groups
    FROM (
        SELECT
            report_date,
            client_hash_id,
            content_hash_id,
            COUNT(*) AS row_count
        FROM march_raw
        GROUP BY
            report_date,
            client_hash_id,
            content_hash_id
        HAVING COUNT(*) > 1
    ) AS duplicates
""").df()

display(grain_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,duplicate_key_groups
0,0


In [18]:
# Verify the size and date range of the March partition.
window_check = con.sql("""
    SELECT
        COUNT(*) AS total_rows,
        MIN(report_date) AS first_date,
        MAX(report_date) AS last_date,
        COUNT(DISTINCT report_date) AS distinct_dates,
        COUNT(DISTINCT client_hash_id) AS client_count,
        COUNT(*) FILTER (
            WHERE report_date IS NULL
               OR client_hash_id IS NULL
               OR content_hash_id IS NULL
        ) AS rows_with_missing_keys
    FROM march_raw
""").df()

display(window_check)

,total_rows,first_date,last_date,distinct_dates,client_count,rows_with_missing_keys
0,9841378,2026-03-01,2026-03-31,31,55,0


In [19]:
# Check how many rows have usable measurements.
availability_check = con.sql("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS gsc_available_rows,
        COUNT(*) FILTER (
            WHERE ga4_data_available IS TRUE
        ) AS ga4_available_rows,
        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
              AND ga4_data_available IS TRUE
        ) AS both_available_rows
    FROM march_raw
""").df()

display(availability_check)

,total_rows,gsc_available_rows,ga4_available_rows,both_available_rows
0,9841378,3611061,413966,364347


In [20]:
import os
from google.colab import userdata
from huggingface_hub import HfApi

# Read the token privately from Colab Secrets.
token = userdata.get("HF_TOKEN").strip()
os.environ["HF_TOKEN"] = token

try:
    account = HfApi(token=token).whoami()
    print("Token is valid.")
    print("Hugging Face username:", account["name"])
except Exception as error:
    print("Verification failed.")
    print("Error type:", type(error).__name__)
    response = getattr(error, "response", None)
    print("HTTP status:", getattr(response, "status_code", None))

Token is valid.
Hugging Face username: omsawarkar


In [21]:
from google.colab import userdata
from huggingface_hub import hf_hub_download
import duckdb

# Read your token privately.
token = userdata.get("HF_TOKEN").strip()

# Download only the March 2026 file.
march_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename=(
        "fact_content_daily_performance/"
        "month=2026-03/data_0.parquet"
    ),
    repo_type="dataset",
    token=token
)

# Open the file and inspect its columns.
con = duckdb.connect()
march = con.read_parquet(
    march_path,
    hive_partitioning=True
)
march.create_view("march_raw")

print("March download successful.")
display(con.sql("DESCRIBE march_raw").df())

March download successful.


,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


## 2. Fields: feature / label / context / excluded

For a diagnostic experiment within my content-review lane, I will explore whether early-March signals help identify items whose average daily search clicks are lower in late March. This observed-change proxy does not establish that a content update is needed or would help.

The input window is 1–20 March 2026. The outcome window is 21–31 March 2026. The conceptual decision point is after the input window’s measurements have become available, before using any outcome-window information. Actual reporting delays would need checking before deployment.

Feature fields — exactly five inputs, calculated using early-window rows where both availability flags are TRUE:

1. Search impressions: total gsc_impressions. Knowable at the decision moment because it uses only available input-window observations.
2. Search clicks: total gsc_clicks. Knowable for the same reason.
3. Search CTR: total clicks divided by total impressions, left missing when impressions are zero. Knowable from input-window observations.
4. Website sessions: total ga4_sessions. Knowable from available input-window analytics.
5. Engagement rate: total ga4_engaged_sessions divided by total ga4_sessions, left missing when sessions are zero. Knowable from input-window analytics.

Label/proxy — late_decline: 1 when average daily search clicks in the outcome window are at least 20% below the input-window average; otherwise 0. Both averages use GSC-available days. The 20% threshold is provisional. Items require at least 10 eligible input days, at least 5 GSC-available outcome days, and a positive input-window click average.

Context — client_hash_id and content_hash_id identify and group items. report_date defines windows. Availability flags and observed-day counts determine eligibility. These are not among the five model inputs.

Excluded — all outcome-window measurements are excluded from features because they reveal later information. IDs are excluded from model inputs. Unavailable measurements are excluded from calculations. June 2026 remains untouched.

The later experiment will compare a model with a simple baseline on held-out clients. Predicting this proxy is a diagnostic exercise, not validation of my earlier Precision@20 human-review goal.

In [22]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

I verified three facts about the March 2026 partition:

1. Grain: There are zero duplicate groups for report_date, client_hash_id, and content_hash_id. There are also zero rows with missing values in these key fields.

2. Size and time window: The partition contains 9,841,378 daily rows, covering 1–31 March 2026 across 55 pseudonymized clients. This does not mean that every content item has observations on all 31 days.

3. Availability: Filtering with gsc_data_available IS TRUE retains 3,611,061 rows. Filtering with ga4_data_available IS TRUE retains 413,966 rows. Requiring both flags to be TRUE retains 364,347 rows.

For the initial five-feature experiment, I will use rows where both sources are available. This restricts coverage, so results will describe this eligible subset rather than all clients or content items. Unavailable measurements will not be treated as genuine zeros.

In [23]:
# Build five early-window features and a later observed-change proxy.
feature_frame = con.sql("""
WITH early_features AS (
    SELECT
        client_hash_id,
        content_hash_id,
        COUNT(*) AS eligible_input_days,

        SUM(gsc_impressions) AS search_impressions,
        SUM(gsc_clicks) AS search_clicks,

        SUM(gsc_clicks) * 1.0
            / NULLIF(SUM(gsc_impressions), 0) AS search_ctr,

        SUM(ga4_sessions) AS website_sessions,

        SUM(ga4_engaged_sessions) * 1.0
            / NULLIF(SUM(ga4_sessions), 0) AS engagement_rate

    FROM march_raw
    WHERE report_date BETWEEN DATE '2026-03-01'
                          AND DATE '2026-03-20'
      AND gsc_data_available IS TRUE
      AND ga4_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
),

early_clicks AS (
    SELECT
        client_hash_id,
        content_hash_id,
        AVG(gsc_clicks) AS early_daily_clicks
    FROM march_raw
    WHERE report_date BETWEEN DATE '2026-03-01'
                          AND DATE '2026-03-20'
      AND gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
),

late_clicks AS (
    SELECT
        client_hash_id,
        content_hash_id,
        COUNT(*) AS outcome_days,
        AVG(gsc_clicks) AS late_daily_clicks
    FROM march_raw
    WHERE report_date BETWEEN DATE '2026-03-21'
                          AND DATE '2026-03-31'
      AND gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
)

SELECT
    f.*,
    e.early_daily_clicks,
    l.outcome_days,
    l.late_daily_clicks,
    CASE
        WHEN l.late_daily_clicks <= 0.8 * e.early_daily_clicks
        THEN 1
        ELSE 0
    END AS late_decline

FROM early_features AS f
JOIN early_clicks AS e
    USING (client_hash_id, content_hash_id)
JOIN late_clicks AS l
    USING (client_hash_id, content_hash_id)

WHERE f.eligible_input_days >= 10
  AND l.outcome_days >= 5
  AND e.early_daily_clicks > 0
""").df()

feature_columns = [
    "search_impressions",
    "search_clicks",
    "search_ctr",
    "website_sessions",
    "engagement_rate"
]

# Check the final row definition.
assert not feature_frame.duplicated(
    ["client_hash_id", "content_hash_id"]
).any()

print("One row = one eligible client–content pair")
print("Eligible items:", len(feature_frame))
print("Clients represented:", feature_frame["client_hash_id"].nunique())

# Display features and proxy without displaying IDs.
display(feature_frame[feature_columns + ["late_decline"]].head())

print("Missing values in the five features:")
display(feature_frame[feature_columns].isna().sum().to_frame("missing_count"))

print("Proxy label counts:")
display(
    feature_frame["late_decline"]
    .value_counts()
    .sort_index()
    .rename_axis("late_decline")
    .to_frame("item_count")
)


One row = one eligible client–content pair
Eligible items: 6502
Clients represented: 20


,search_impressions,search_clicks,search_ctr,website_sessions,engagement_rate,late_decline
0,164.0,1.0,0.006098,22.0,0.136364,1
1,12645.0,144.0,0.011388,125.0,0.080000,1
2,4291.0,18.0,0.004195,16.0,0.000000,0
3,245.0,3.0,0.012245,26.0,0.076923,1
4,682.0,4.0,0.005865,45.0,0.155556,0


Missing values in the five features:


,missing_count
search_impressions,0
search_clicks,0
search_ctr,0
website_sessions,0
engagement_rate,0


Proxy label counts:


,item_count
late_decline,
0,3134
1,3368


In [24]:
%pip -q install scikit-learn

import pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from sklearn.tree import DecisionTreeClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score

# Only these five columns are honest model inputs.
X = feature_frame[feature_columns].copy()
y = feature_frame["late_decline"].astype(int)
groups = feature_frame["client_hash_id"]

# Hold out entire clients so a client's items cannot cross the split.
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=42
)
train_idx, test_idx = next(splitter.split(X, y, groups))

assert set(groups.iloc[train_idx]).isdisjoint(
    set(groups.iloc[test_idx])
)

print("Training items:", len(train_idx))
print("Test items:", len(test_idx))
print("Training clients:", groups.iloc[train_idx].nunique())
print("Test clients:", groups.iloc[test_idx].nunique())

results = []

def evaluate(name, model, inputs):
    model.fit(inputs.iloc[train_idx], y.iloc[train_idx])
    predictions = model.predict(inputs.iloc[test_idx])

    results.append({
        "experiment": name,
        "accuracy": accuracy_score(y.iloc[test_idx], predictions),
        "balanced_accuracy": balanced_accuracy_score(
            y.iloc[test_idx], predictions
        )
    })

# Baseline: always predict the most common training label.
evaluate(
    "Majority baseline",
    DummyClassifier(strategy="most_frequent"),
    X
)

# Deliberate mistake: copy the answer into an input column.
X_leaky = X.copy()
X_leaky["LEAK_answer_copy"] = y

evaluate(
    "Deliberately leaked tree",
    DecisionTreeClassifier(max_depth=3, random_state=42),
    X_leaky
)

# Remove the leaked column and train a fresh model.
X_clean = X_leaky.drop(columns="LEAK_answer_copy")
del X_leaky

assert list(X_clean.columns) == feature_columns

evaluate(
    "Honest five-feature tree",
    DecisionTreeClassifier(max_depth=3, random_state=42),
    X_clean
)

experiment_results = pd.DataFrame(results)
display(experiment_results.round(4))

Training items: 6007
Test items: 495
Training clients: 15
Test clients: 5


,experiment,accuracy,balanced_accuracy
0,Majority baseline,0.3495,0.5000
1,Deliberately leaked tree,1.0000,1.0000
2,Honest five-feature tree,0.6081,0.5035


## 4. Data limits

### Scope and coverage

The final experimental table contains 6,502 eligible client–content pairs across 28 clients. None of the five features is missing in this resulting subset. There are 3,368 positive decline-proxy labels and 3,134 negative labels.

Requiring both GSC and GA4 availability substantially restricts coverage. Results describe this selected subset, not the whole warehouse. Content items have different observation histories; unavailable or absent days must not be interpreted as zero activity. Feature totals also depend on the number of observed input days.

### Deliberate leakage experiment

I held out entire clients: 6,007 items from 15 clients were used for training, and 495 items from 13 different clients were used for testing.

The majority baseline achieved accuracy 0.3495 and balanced accuracy 0.5000. Adding an exact copy of the outcome label as an input produced accuracy and balanced accuracy of 1.0000. This perfect result is invalid because the input directly reveals the answer.

I removed the leaked column and trained a fresh tree using only the five early-window features. Its accuracy was 0.6081 and balanced accuracy was 0.5035. The balanced accuracy is close to chance, so this experiment does not establish useful predictive skill. Different label proportions across training and test clients help explain why ordinary accuracy is misleading here.

### What this cannot establish

The proxy measures a thresholded change in observed daily clicks. It does not establish that a content update is needed or that an update would improve performance. Short windows, low click counts, differing coverage, and normal traffic variation can affect the label.

This is one exploratory client-grouped split, not a robust estimate of future performance. The input and outcome periods differ in length and weekday composition. Actual reporting delays would also need checking.

June 2026 remains unused. No causal claim, validated refresh recommendation, or human-review Precision@20 result is made.

In [25]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.